# 예제 06. 금융업무 프롬프트 작성과 로컬 LLM 활용
### 생성형 AI에게 금융업무를 지시하는 방법 · 2일 차 · 60분

> **실행 방법**: 위에서부터 순서대로 실행하세요. 셀을 클릭하고 `Shift + Enter`. AI 답변이 나오는 셀은 몇 초~수십 초 걸립니다. 셀 왼쪽이 `[*]`이면 아직 실행 중입니다.

| 항목 | 내용 |
|---|---|
| 실습 환경 | Jupyter Notebook · Ollama · Qwen3 4B Instruct (이 PC 안에서 실행) |
| 데이터 | `data/bank_customers.csv` (1일 차 고객 데이터 45,211건) |
| 오늘의 결과물 | 금융업무 프롬프트 템플릿 3종 + 프롬프트 비교 기록 (`outputs/ex06_프롬프트_비교기록.md`) |

**오늘 해결할 질문**: "같은 데이터인데, 어떻게 지시하면 AI가 업무에 쓸 만한 답을 줄까?"

## 0. 실습 준비

FinanceAgent를 불러오고, 이 PC의 로컬 AI(Ollama)가 켜져 있는지 확인합니다.
`AI 연결 확인: qwen3:4b-instruct 준비 완료`가 나오면 성공입니다.

In [ ]:
import json, urllib.request
from pathlib import Path
from finance_agent import FinanceAgent
import llm_config

DATA = "data/bank_customers.csv"
models = [m["name"] for m in json.load(urllib.request.urlopen("http://127.0.0.1:11434/api/tags"))["models"]]
print("데이터 파일:", Path(DATA).is_file())
print("AI 연결 확인:", llm_config.OLLAMA_MODEL, "준비 완료" if llm_config.OLLAMA_MODEL in models else "→ 모델이 없습니다. 강사에게 알려 주세요")
agent = FinanceAgent(provider="ollama")

## 06-1. 클라우드 AI와 로컬 AI는 무엇이 다를까?

| | 클라우드 AI (ChatGPT, Gemini 등) | 로컬 AI (오늘: Ollama + Qwen) |
|---|---|---|
| 실행 위치 | 회사 밖 서버 | **이 PC 안** |
| 입력한 데이터 | 인터넷으로 외부에 전송됨 | PC 밖으로 나가지 않음 |
| 비용 | 사용량만큼 과금(API) | 무료 (PC 성능만 필요) |
| 성능 | 매우 큰 모델, 문장력 뛰어남 | 작은 모델(4B), 짧고 정해진 일에 적합 |
| 인터넷 | 필요 | 모델을 받은 뒤에는 불필요 |

> ⚠️ 금융 내부자료와 고객정보는 **승인 없이 외부 AI에 입력하지 않습니다.** 오늘 데이터는 공개된 해외 데이터라 실습에 쓸 수 있습니다.

## 06-2. 로컬 AI에게 첫 질문

`agent.ask("질문")`은 질문을 AI에게 그대로 보내고 답을 보여 줍니다. 아래 질문을 바꿔서도 실행해 보세요.

In [ ]:
agent.ask("정기예금과 정기적금의 차이를 저축은행 고객에게 설명하듯 3줄로 정리해 주세요.")

## 06-3. AI에게 숫자를 물으면?

1일 차에 직접 계산한 "저축은행 79개사 총자산 합계(2026년 3월 말)"를 AI에게 물어봅니다. 1일 차 정답은 **1,192,698억 원**이었습니다.

In [ ]:
agent.ask("2026년 3월 말 기준 국내 저축은행 79개사의 총자산 합계는 얼마인가요? 숫자로 답해 주세요.")

> 💡 **핵심 정리** — AI는 모르는 숫자도 그럴듯하게 지어낼 수 있습니다(이를 **환각**이라고 합니다).
> 그래서 오늘의 원칙은 **"계산은 Python(Pandas)이, 해석과 문장은 AI가"** 입니다.

## 06-4. FinanceAgent는 어떻게 일하나?

오늘 쓰는 FinanceAgent는 정해진 순서대로만 움직이는 **업무 흐름(워크플로)형 에이전트**입니다.

```
[① 입력 검증] → [② 금융업무 기능] → [③ 프롬프트 + AI] → [④ 결과보기]
 파일이 있나?     Pandas로 가입률 계산    계산 결과를 AI에게 주고 해석    화면·Excel·차트·Word
```

예제마다 ②·③·④에서 쓰는 함수만 바뀝니다. 교육생은 **질문 · 데이터 파일 · 예제 번호**만 고릅니다.

## 06-5. AI에게 넘어가는 숫자부터 보기

②번 단계(`run_function`)가 계산한 결과입니다. AI는 이 숫자를 받아서 해석만 합니다.

In [ ]:
from exercises import ex06_prompt

질문 = "저축은행 고객의 연령대별 정기예금 가입률을 분석한 결과입니다. 주요 특징을 요약하고, 예금 유치 전략을 검토하기 위한 보고서 문장으로 작성해주세요."
요약 = ex06_prompt.run_function(DATA, 질문)
요약

## 06-6. 프롬프트 ⓪ "단순" — 질문과 숫자만 던지기

역할·목적·기준·형식 없이 질문과 숫자만 보냅니다. 결과를 잘 봐 두세요.

In [ ]:
ex06_prompt.TEMPLATE_NAME = "단순"
결과 = agent.run("ex06", DATA, 질문)

> 💬 AI 답변의 문장은 PC와 모델에 따라 조금씩 다릅니다. 표·숫자·파일은 Python이 계산하므로 모두 같게 나옵니다.

## 06-7. 좋은 업무 프롬프트의 4요소

| 요소 | 뜻 | 예 |
|---|---|---|
| **역할** | AI가 누구처럼 답할지 | 저축은행 수신기획팀 데이터 분석 담당자 |
| **업무 목적** | 결과를 어디에 쓸지 | 정기예금 유치 전략 회의 자료 |
| **분석 기준** | 무엇을 비교·주의할지 | 전체 평균과 비교, 고객수 적은 집단 주의, 숫자 지어내지 않기 |
| **출력 형식** | 어떤 모양으로 받을지 | 핵심 요약 2문장 + 글머리 3개 + 유의사항 |

아래는 이 4요소를 갖춘 **템플릿 ① 분석 요약형**입니다.

In [ ]:
print(ex06_prompt.TEMPLATES["분석요약"])

## 06-8. 템플릿 ① 분석 요약형으로 다시 실행

06-6의 "단순" 결과와 비교해 보세요. 형식이 정해졌나요? 숫자가 정확한가요?

In [ ]:
ex06_prompt.TEMPLATE_NAME = "분석요약"
결과 = agent.run("ex06", DATA, 질문)

## 06-9. 템플릿 ② 보고서 문장형 · 템플릿 ③ 검토 의견형

같은 데이터로 **용도**만 바꿉니다.

In [ ]:
ex06_prompt.TEMPLATE_NAME = "보고서문장"
결과 = agent.run("ex06", DATA, 질문)

In [ ]:
ex06_prompt.TEMPLATE_NAME = "검토의견"
결과 = agent.run("ex06", DATA, 질문)

> 💡 **핵심 정리** — 데이터는 그대로인데 **역할과 출력 형식**만 바꿔도 결과물의 쓰임새가 달라집니다.
> 결과 아래의 **숫자 검증** 줄은 AI 문장 속 숫자가 계산 결과에 있는지 Python이 자동으로 확인한 것입니다.

## 06-10. AI에게 실제로 전달된 프롬프트 보기

`결과["prompt"]`에 방금 AI에게 보낸 전체 지시문이 들어 있습니다. 질문 + 템플릿 + 계산 결과가 합쳐진 모습을 확인합니다.

In [ ]:
print(결과["prompt"])

## 06-11. 🔧 나만의 프롬프트 템플릿 만들기 (실습)

준비된 예시에서 **역할 · 업무 목적 · 출력 형식** 줄만 우리 업무에 맞게 고칩니다. `{question}`과 `{data}` 두 자리는 그대로 두세요(질문과 계산 결과가 들어갈 자리).

In [ ]:
ex06_prompt.TEMPLATES["나의템플릿"] = """[역할] 당신은 저축은행 영업점의 고객상담 팀장입니다.
[업무 목적] 창구 직원들이 아침 회의에서 공유할 정기예금 안내 포인트를 정리합니다.
[업무 요청] {question}

[분석 데이터] 아래 숫자만 사용하세요.
{data}

[분석 기준] 가입률이 높은 고객군의 특징을 쉬운 말로 설명하고, 특정 고객을 차별하는 표현은 쓰지 마세요.
[출력 형식] 글머리 3개, 각 1문장, 한국어"""

ex06_prompt.TEMPLATE_NAME = "나의템플릿"
결과 = agent.run("ex06", DATA, 질문)

## 06-12. [선택] 질문을 바꾸면 계산도 바뀐다

질문에 "직업", "잔액", "대출", "캠페인" 같은 낱말이 있으면 ②번 단계가 그 기준으로 가입률을 계산합니다. (AI가 아니라 정해진 낱말 목록으로 고릅니다.)

In [ ]:
ex06_prompt.TEMPLATE_NAME = "분석요약"
결과 = agent.run("ex06", DATA, "직업군별 정기예금 가입률의 주요 특징을 요약해 주세요.")

## 06-13. [선택] 프롬프트 비교 기록 확인

실행할 때마다 `outputs/ex06_프롬프트_비교기록.md`에 템플릿 이름 · 질문 · 답변이 쌓입니다. 아래 셀로 마지막 부분을 보거나, 파일 탐색기에서 열어 보세요.

In [ ]:
기록 = Path("outputs/ex06_프롬프트_비교기록.md").read_text(encoding="utf-8")
print(기록[-1500:])

## 06-14. [선택] Gemini · OpenAI 연결 (개인 API 키가 있는 교육생만)

FinanceAgent는 `provider`만 바꾸면 다른 AI로 연결됩니다. API 키는 화면에 보이지 않게 입력합니다.

> ⚠️ 외부 AI에는 **공개 데이터만** 입력합니다. 개인용 ChatGPT·Gemini 유료 구독과 API 이용 권한은 별개입니다(API는 별도 결제).

In [ ]:
import os
from getpass import getpass

# os.environ["GEMINI_MODEL"] = "여기에 모델 ID"        # Google AI Studio에서 확인한 이름
# os.environ["GEMINI_API_KEY"] = getpass("Gemini API 키: ")
# FinanceAgent(provider="gemini").ask("정기예금을 한 문장으로 설명해 주세요.")

## 06-15. 해석할 때 꼭 기억할 점 ⚠️

- AI 문장은 **초안**입니다. 숫자는 원본 표와, 표현은 내부 기준과 대조한 뒤 씁니다.
- 작은 로컬 모델(4B)은 긴 글·복잡한 추론에 약합니다. 지시는 **짧고 구체적으로**, 출력 형식은 **정해서** 줍니다.
- "숫자를 지어내지 마세요"라고 써도 지어낼 수 있습니다. 그래서 **숫자 검증**을 Python으로 따로 합니다.

## ✅ 오늘 배운 것

| 하고 싶은 일 | 코드 |
|---|---|
| AI에게 바로 질문 | `agent.ask("질문")` |
| 예제 모듈 실행 | `agent.run("ex06", DATA, 질문)` |
| 템플릿 바꾸기 | `ex06_prompt.TEMPLATE_NAME = "보고서문장"` |
| 새 템플릿 추가 | `ex06_prompt.TEMPLATES["이름"] = """...{question}...{data}..."""` |
| 보낸 프롬프트 보기 | `결과["prompt"]` |

## 📝 복습 미션

> 아래 셀에서 **`# ←` 표시가 있는 줄만** 바꿔 실행하세요. 막히면 맨 아래 **정답 셀**을 실행해 결과를 확인합니다.

**미션 1.** `검토의견` 템플릿으로 "잔액 구간별 정기예금 가입률"을 질문해 보세요.
**이렇게 나오면 성공!** 06-5처럼 `run_function`으로 확인하면 `잔액구간별 가입률`에 마이너스 5.6% ~ 5,000 초과 15.5%가 들어 있음

**미션 2.** 06-11의 나의 템플릿에서 **출력 형식**만 "표 형태(고객군 | 특징 | 안내 포인트)"로 바꿔 다시 실행해 보세요.
**이렇게 나오면 성공!** 답변이 표 모양으로 바뀜 (모양은 AI에 따라 조금씩 다름)

**다음 예제 전에 이것만은 꼭!**
- [ ] 계산은 Python, 해석은 AI
- [ ] 프롬프트 4요소: 역할 · 업무 목적 · 분석 기준 · 출력 형식

In [ ]:
# 미션 1
ex06_prompt.TEMPLATE_NAME = "분석요약"     # ← "검토의견"으로
결과 = agent.run("ex06", DATA, "연령대별 정기예금 가입률을 검토해 주세요.")   # ← "연령대별"을 "잔액 구간별"로

### 정답 (먼저 위 셀을 바꿔 실행해 본 뒤 확인하세요)

In [ ]:
# 미션 1 정답
print(ex06_prompt.run_function(DATA, "잔액 구간별 정기예금 가입률")["잔액구간별 가입률"])
ex06_prompt.TEMPLATE_NAME = "검토의견"
결과 = agent.run("ex06", DATA, "잔액 구간별 정기예금 가입률을 검토해 주세요.")

---
### ▶ 다음 예제 07. Excel 업무 자동화 에이전트
오늘 익힌 프롬프트를 이용해, 고객군별 가입률 집계 → AI 해석 → **서식이 갖춰진 Excel 보고서**까지 한 번에 만듭니다.